# 1. Import Libraries

In [1]:
import math
from pulp import LpProblem, LpVariable, LpStatus, lpSum, LpMaximize, LpInteger, LpContinuous, value
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
import pulp
import random
import warnings
from IPython.display import display
warnings.filterwarnings('ignore')


# 2. Read Data Files 

In [2]:
bom = pd.read_csv('data/bill_of_materials.csv')
cakes = pd.read_csv('data/cakes.csv')
channels = pd.read_csv('data/channels.csv')
ingredients = pd.read_csv('data/ingredients.csv')
demand_params = pd.read_csv('data/instructor_demand_competition.csv')
wages_energy = pd.read_csv('data/wages_energy.csv')

# 3. Extract Wages and Cost Parameters

In [3]:
param_map = dict(zip(wages_energy.parameter, wages_energy.value))
prep_wage = param_map['prep_wage_usd_per_hour']
oven_wage = param_map['oven_wage_usd_per_hour']
pack_wage = param_map['pack_wage_usd_per_hour']
oven_rental = param_map['oven_rental_usd_per_hour']
oven_cost = param_map['oven_cost_usd_per_hour']
budget = param_map['budget_usd']

cake_info = cakes.set_index('cake_id')
channel_info = channels.set_index('channel')
ingredient_cost = ingredients.set_index('ingredient')['unit_cost_usd'].to_dict()

ingredient_cols = [c for c in bom.columns if c not in ['cake_id','name']]
usage = bom.set_index('cake_id')[ingredient_cols]

# 4. Price Placeholders By Market and by Cake

In [4]:
pairs = demand_params[['ID','cake_name','channel']].drop_duplicates().rename(columns={'ID':'cake_id'})
cake_market_prices = {(row.cake_id, row.channel): None for row in pairs.itertuples(index=False)}
cake_market_price_table = pairs.copy()
cake_market_price_table['price'] = None
display(cake_market_price_table.sort_values(['cake_id','channel']).reset_index(drop=True))
summaries = {}
try:
    summaries['cake_market_price_placeholders'] = cake_market_price_table
except Exception:
    pass



,cake_id,cake_name,channel,price
0,1,Triple Chocolate Layer Cake,Local,None
1,1,Triple Chocolate Layer Cake,Online,None
2,1,Triple Chocolate Layer Cake,Supermarket,None
3,2,Victoria Sponge,Local,None
4,2,Victoria Sponge,Online,None
5,2,Victoria Sponge,Supermarket,None
6,3,Red Velvet Cake,Local,None
7,3,Red Velvet Cake,Online,None
8,3,Red Velvet Cake,Supermarket,None
9,4,Lemon Drizzle Cake,Local,None


# Fill in prices

In [5]:

def compute_unit_costs_excl_transport():
    ing_cost = (usage * pd.Series(ingredient_cost)).sum(axis=1)
    labor_unit = (cake_info['prep_min_per_unit'] * prep_wage + cake_info['pack_min_per_unit'] * pack_wage) / 60.0
    oven_unit = (cake_info['oven_min_per_batch'] * (oven_wage + oven_rental + oven_cost) / 60.0) / cake_info['batch_size_units']
    packaging = cake_info['packaging_cost_per_unit_usd']
    cost = (ing_cost + labor_unit + oven_unit + packaging)
    return cost

def suggest_prices(fill=True, round_to=2):
    base_cost = compute_unit_costs_excl_transport()
    trans_cost = channel_info['transport_cost_per_unit_usd']

    dp = demand_params.rename(columns={'ID': 'cake_id'}).copy()
    dp['unit_cost'] = dp['cake_id'].map(base_cost) + dp['channel'].map(trans_cost)
  
    dp['p_suggest'] = 0.5 * (dp['unit_cost'] + dp['alpha'] / dp['beta'])

    if fill:
        price_map = {(int(r.cake_id), r.channel): round(float(r.p_suggest), round_to) for r in dp.itertuples(index=False)}
        cake_market_price_table['price'] = cake_market_price_table.apply(lambda r: price_map[(int(r.cake_id), r.channel)], axis=1)
        cake_market_price_table['unit_cost'] = cake_market_price_table.apply(lambda r: float(base_cost.loc[int(r.cake_id)] + trans_cost.loc[r.channel]), axis=1)
        cake_market_price_table['suggested'] = cake_market_price_table['price']
    return dp[['cake_id', 'channel', 'unit_cost', 'p_suggest']]

def set_price(cake_id: int, channel: str, price: float):
    mask = (cake_market_price_table['cake_id'] == int(cake_id)) & (cake_market_price_table['channel'] == channel)
    if not mask.any():
        raise ValueError(f'Pair (cake_id={cake_id}, channel={channel}) not found.')
    cake_market_price_table.loc[mask, 'price'] = float(price)

def set_prices(price_dict: dict):
    for (i, j), p in price_dict.items():
        set_price(i, j, p)

def validate_prices():
    missing = cake_market_price_table['price'].isnull()
    if missing.any():
        print('Missing prices for:', cake_market_price_table[missing][['cake_id', 'channel']].to_dict('records'))
        return False
    return True

_ = suggest_prices(fill=True)
display(cake_market_price_table.sort_values(['cake_id', 'channel']).reset_index(drop=True))

,cake_id,cake_name,channel,price,unit_cost,suggested
0,1,Triple Chocolate Layer Cake,Local,19.72,7.919222,19.72
1,1,Triple Chocolate Layer Cake,Online,16.74,8.319222,16.74
2,1,Triple Chocolate Layer Cake,Supermarket,12.15,8.119222,12.15
3,2,Victoria Sponge,Local,19.34,5.621083,19.34
4,2,Victoria Sponge,Online,15.28,6.021083,15.28
5,2,Victoria Sponge,Supermarket,9.97,5.821083,9.97
6,3,Red Velvet Cake,Local,20.19,8.574000,20.19
7,3,Red Velvet Cake,Online,15.17,8.974000,15.17
8,3,Red Velvet Cake,Supermarket,12.40,8.774000,12.40
9,4,Lemon Drizzle Cake,Local,20.44,6.644583,20.44


# 5. Simple Profit Function and Demand

We treat prices as inputs taken from `cake_market_price_table`.

Objective (maximize):

Π = Σ_{i,j} P_{ij} · s_{ij} − C_ingredients − C_labor − C_utilities − C_transportation

with

• Demand: D_{ij} = α_{ij} − β_{ij} P_{ij}

• Sales bounded by production and demand: s_{ij} = min(D_{ij}, y_{ij}).

We linearize the min via constraints: s_{ij} ≤ y_{ij} and s_{ij} ≤ D_{ij}(P_{ij}). Prices must be provided for every (cake, channel).

In [6]:
def _prices_from_table(price_table: pd.DataFrame):
    if price_table['price'].isnull().any():
        missing = price_table[price_table['price'].isnull()][['cake_id','channel']]
        raise ValueError(f"Please set a price for every (cake_id, channel). Missing: {missing.to_dict('records')}")
    return {(int(r.cake_id), r.channel): float(r.price) for r in price_table.itertuples(index=False)}

def _demand_from_prices(prices: dict):
    ab = {(int(r.ID), r.channel): (float(r.alpha), float(r.beta)) for r in demand_params.itertuples(index=False)}
    D = {}
    for (i,j), p in prices.items():
        a, b = ab[(i,j)]
        D[(i,j)] = max(0.0, a - b * p)
    return D



def build_simple_profit_model(price_table: pd.DataFrame):
    prices = _prices_from_table(price_table)
    Dcap = _demand_from_prices(prices)

    prob = LpProblem('SimpleProfit', LpMaximize)

    markets = list(channel_info.index)
    cake_ids = list(cake_info.index)
    ingredients_list = list(usage.columns)

    # Decision variables
    y = {(i,j): LpVariable(f'y_({i},{j})', lowBound=0, cat=LpInteger) for i in cake_ids for j in markets}
    s = {(i,j): LpVariable(f's_({i},{j})', lowBound=0, cat=LpInteger) for i in cake_ids for j in markets}
    b = {i: LpVariable(f'batch_{i}', lowBound=0, cat=LpInteger) for i in cake_ids}
    y_ing = {k: LpVariable(f'y_{k}', lowBound=0, cat=LpContinuous) for k in ingredients_list}
    z_make = {i: LpVariable(f'z_make_{i}', cat='Binary') for i in cake_ids}

    # Helpers
    total_units_by_cake = {i: lpSum(y[(i,j)] for j in markets) for i in cake_ids}
    D_total_by_cake = {i: sum(Dcap[(i,j)] for j in markets) for i in cake_ids}

    # Costs
    C_ingredients = lpSum(ingredient_cost[k] * y_ing[k] for k in ingredients_list)
    prep_minutes = lpSum(cake_info.loc[i, 'prep_min_per_unit'] * total_units_by_cake[i] for i in cake_ids)
    pack_minutes = lpSum(cake_info.loc[i, 'pack_min_per_unit'] * total_units_by_cake[i] for i in cake_ids)
    oven_minutes = lpSum(cake_info.loc[i, 'oven_min_per_batch'] * b[i] for i in cake_ids)
    C_labor = (prep_minutes * prep_wage + pack_minutes * pack_wage + oven_minutes * oven_wage) / 60.0
    
    # Utilities: oven rental + electricity + packaging materials
    C_utilities = (oven_minutes * (oven_rental + oven_cost)) / 60.0 + lpSum(cake_info.loc[i, 'packaging_cost_per_unit_usd'] * total_units_by_cake[i] for i in cake_ids)
    C_transportation = lpSum(channel_info.loc[j, 'transport_cost_per_unit_usd'] * s[(i,j)] for i in cake_ids for j in markets)

    # Revenue
    Revenue = lpSum(prices[(i,j)] * s[(i,j)] for i in cake_ids for j in markets)

    prob += Revenue - C_ingredients - C_labor - C_utilities - C_transportation, 'Profit'

    # Sales bounded by production and demand
    for i in cake_ids:
        for j in markets:
            prob += s[(i,j)] <= y[(i,j)], f's_le_prod_{i}_{j}'
            prob += s[(i,j)] <= Dcap[(i,j)], f's_le_demand_{i}_{j}'

    # Channel service capacities (by sales)
    for j in markets:
        cap = float(channel_info.loc[j, 'service_cap_per_week'])
        prob += lpSum(s[(i,j)] for i in cake_ids) <= cap, f'channel_cap_{j}'

    # Ingredient purchase covers usage
    for k in ingredients_list:
        prob += lpSum(usage.loc[i, k] * total_units_by_cake[i] for i in cake_ids) <= y_ing[k], f'ing_{k}_balance'

    # Batches enforce batch sizes
    for i in cake_ids:
        prob += total_units_by_cake[i] == b[i] * int(cake_info.loc[i, 'batch_size_units']), f'batch_size_{i}'

    # Minimum units if made (activation) using binary z_make and big-M from total demand cap
    for i in cake_ids:
        M = D_total_by_cake[i] if D_total_by_cake[i] > 0 else 1000
        min_units = int(cake_info.loc[i, 'minimum_units_if_made'])
        prob += total_units_by_cake[i] >= min_units * z_make[i], f'min_units_{i}'
        prob += total_units_by_cake[i] <= M * z_make[i], f'M_cap_{i}'

    # Budget (investment) constraint: ingredients + labor + utilities spend <= budget
    prob += C_ingredients + C_labor + C_utilities <= budget, 'budget'

    context = {
        'prices': prices, 'Dcap': Dcap, 'vars_y': y, 'vars_s': s, 'vars_batches': b, 'vars_ing': y_ing, 'vars_z': z_make,
        'Revenue': Revenue, 'C_ingredients': C_ingredients, 'C_labor': C_labor, 'C_utilities': C_utilities, 'C_transportation': C_transportation
    }
    return prob, context


try:
    _prob_preview, _ctx = build_simple_profit_model(cake_market_price_table)
    print('Model ready. Objective:', _prob_preview.objective) 
except ValueError as e:
    print('Set prices in cake_market_price_table before building the model.', e)

Model ready. Objective: -7.833333333333334*batch_1 - 7.206666666666667*batch_10 - 6.11*batch_2 - 7.5200000000000005*batch_3 - 11.436666666666667*batch_4 - 15.196666666666667*batch_5 - 6.58*batch_6 - 10.653333333333334*batch_7 - 12.533333333333333*batch_8 - 7.206666666666667*batch_9 + 19.52*s_(1,Local) + 16.139999999999997*s_(1,Online) + 11.75*s_(1,Supermarket) + 18.0*s_(10,Local) + 15.32*s_(10,Online) + 10.87*s_(10,Supermarket) + 19.14*s_(2,Local) + 14.68*s_(2,Online) + 9.57*s_(2,Supermarket) + 19.990000000000002*s_(3,Local) + 14.57*s_(3,Online) + 12.0*s_(3,Supermarket) + 20.240000000000002*s_(4,Local) + 14.3*s_(4,Online) + 10.73*s_(4,Supermarket) + 22.28*s_(5,Local) + 17.32*s_(5,Online) + 14.09*s_(5,Supermarket) + 20.580000000000002*s_(6,Local) + 14.93*s_(6,Online) + 10.83*s_(6,Supermarket) + 20.080000000000002*s_(7,Local) + 13.64*s_(7,Online) + 8.98*s_(7,Supermarket) + 18.44*s_(8,Local) + 13.97*s_(8,Online) + 11.52*s_(8,Supermarket) + 18.78*s_(9,Local) + 14.51*s_(9,Online) + 11.18*s_

Profit Found:

In [7]:
# Solve the model and present profit found
prob, ctx = build_simple_profit_model(cake_market_price_table)

# Try preferred solvers, fall back to default if unavailable
solver = None
try:
    solver = pulp.PULP_CBC_CMD(msg=False)
    _ = prob.solve(solver)
except Exception:
    _ = prob.solve()

status = LpStatus[prob.status]
profit = float(value(prob.objective))
rev = float(value(ctx['Revenue']))
c_ing = float(value(ctx['C_ingredients']))
c_lab = float(value(ctx['C_labor']))
c_utl = float(value(ctx['C_utilities']))
c_trn = float(value(ctx['C_transportation']))

print(f'Status: {status}')
print(f'Profit: ${profit:,.2f}')
print(f'Revenue: ${rev:,.2f}')
print('Costs:')
print(f'  Ingredients: ${c_ing:,.2f}')
print(f'  Labor:       ${c_lab:,.2f}')
print(f'  Utilities:   ${c_utl:,.2f}')
print(f'  Transport:   ${c_trn:,.2f}')

Status: Optimal
Profit: $5,127.54
Revenue: $9,363.25
Costs:
  Ingredients: $2,315.66
  Labor:       $1,060.88
  Utilities:   $620.77
  Transport:   $238.40
